# 05 - Tuning, Evaluasi Akhir, dan Pemilihan Model
**Proyek:** Hotel Booking Cancellation Prediction | **Tahap:** M6 | **PRD:** v1.1

Aturan main (PRD 9.1 dan 9.4):
1. **Tuning dan pemilihan threshold hanya memakai data train** (cross-validation).
2. Model final dipilih **dari hasil CV di data train**.
3. **Data test dipakai satu kali** di bagian 4 untuk evaluasi akhir; tidak dipakai untuk memilih apa pun.

Library: `pip install scikit-learn xgboost joblib pandas numpy matplotlib seaborn`

In [ ]:
import os, glob, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib
from scipy.stats import loguniform
from sklearn.model_selection import (train_test_split, StratifiedKFold, RandomizedSearchCV, GridSearchCV, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve, precision_recall_curve, brier_score_loss)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
os.makedirs('models', exist_ok=True); os.makedirs('reports/figures', exist_ok=True)

cands = (glob.glob('/kaggle/input/**/hotel_bookings.csv', recursive=True) + glob.glob('./hotel_bookings.csv')
         + glob.glob('./data/raw/hotel_bookings.csv') + glob.glob('/mnt/user-data/uploads/hotel_bookings.csv'))
assert cands, 'hotel_bookings.csv tidak ditemukan'
raw = pd.read_csv(cands[0])
df = raw.drop_duplicates(subset=[c for c in raw.columns if c != 'reservation_status_date']).reset_index(drop=True)

TARGET = 'is_canceled'
FEATS = ['lead_time','country','market_segment','total_of_special_requests','customer_type']
train_idx, test_idx = train_test_split(df.index, test_size=0.2, stratify=df[TARGET], random_state=RANDOM_STATE)
X_train, y_train = df.loc[train_idx, FEATS], df.loc[train_idx, TARGET]
X_test,  y_test  = df.loc[test_idx,  FEATS], df.loc[test_idx,  TARGET]   # JANGAN dipakai sebelum bagian 4
print('Train:', X_train.shape, '| Test (disimpan, belum dipakai):', X_test.shape)

## 1. Pipeline dan tuning terkontrol (CV pada data train)

In [ ]:
log_tf = FunctionTransformer(np.log1p, feature_names_out='one-to-one')
def build_prep(linear=False):
    country = Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='Unknown')),
                        ('oh', OneHotEncoder(handle_unknown='infrequent_if_exist', max_categories=11, sparse_output=False))])
    tfs = [('country', country, ['country']),
           ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['market_segment','customer_type'])]
    if linear:
        tfs += [('lead', Pipeline([('log', log_tf), ('sc', StandardScaler())]), ['lead_time']),
                ('req', StandardScaler(), ['total_of_special_requests'])]
    else:
        tfs += [('num', 'passthrough', ['lead_time','total_of_special_requests'])]
    return ColumnTransformer(tfs)

# Catatan: tanpa class_weight. Trade-off Precision/Recall diatur lewat THRESHOLD (bagian 2),
# sehingga probabilitas model tidak digeser oleh pembobotan.
pipes = {
 'LogReg':       Pipeline([('prep', build_prep(True)),  ('model', LogisticRegression(max_iter=2000))]),
 'RandomForest': Pipeline([('prep', build_prep(False)), ('model', RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE))]),
 'XGBoost':      Pipeline([('prep', build_prep(False)), ('model', XGBClassifier(eval_metric='logloss', n_jobs=-1, random_state=RANDOM_STATE))]),
}
spaces = {
 'LogReg': ({'model__C': [0.01, 0.1, 1, 10, 100]}, 'grid'),
 'RandomForest': ({'model__n_estimators': [200, 300], 'model__max_depth': [8, 12, 16, None],
                   'model__min_samples_leaf': [3, 5, 10, 20], 'model__min_samples_split': [2, 10],
                   'model__max_features': ['sqrt', 0.5]}, 8),
 'XGBoost': ({'model__n_estimators': [200, 300, 500], 'model__max_depth': [3, 4, 5, 6, 8],
              'model__learning_rate': [0.03, 0.05, 0.1], 'model__subsample': [0.7, 0.8, 1.0],
              'model__colsample_bytree': [0.7, 0.8, 1.0], 'model__reg_lambda': [1, 5, 10],
              'model__min_child_weight': [1, 5]}, 20),
}
skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
SCORING = {'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'}

searches, rows = {}, []
for name, pipe in pipes.items():
    grid, n_iter = spaces[name]; t0 = time.time()
    if n_iter == 'grid':
        s = GridSearchCV(pipe, grid, scoring=SCORING, refit='pr_auc', cv=skf3, n_jobs=1)
    else:
        s = RandomizedSearchCV(pipe, grid, n_iter=n_iter, scoring=SCORING, refit='pr_auc', cv=skf3,
                               n_jobs=1, random_state=RANDOM_STATE)
    s.fit(X_train, y_train); searches[name] = s
    i = s.best_index_
    rows.append({'model': name, 'CV PR-AUC': s.cv_results_['mean_test_pr_auc'][i], 'CV ROC-AUC': s.cv_results_['mean_test_roc_auc'][i],
                 'detik': time.time()-t0, 'parameter terbaik': {k.replace('model__',''): v for k, v in s.best_params_.items()}})
    print(f'{name} selesai ({time.time()-t0:.0f} dtk)')
cv_table = pd.DataFrame(rows).set_index('model')
pd.set_option('display.max_colwidth', 200)
cv_table.round(4)

**Interpretasi tuning (3-fold CV, data train, metrik utama PR-AUC):**
- Random Forest (PR-AUC 0,6470; ROC-AUC 0,8338) dan XGBoost (0,6464; 0,8344) **hampir identik**; selisih PR-AUC hanya 0,0006, jauh di bawah variasi antar fold. Logistic Regression tertinggal (0,5939; 0,8002) meski sudah di-tuning, yang menunjukkan hubungan fitur dengan target tidak sepenuhnya linear.
- Parameter terbaik Logistic Regression `C=0,01` (regularisasi kuat). Random Forest memilih `max_depth=12` dan `min_samples_leaf=5` (pohon dibatasi agar tidak menghafal), XGBoost memilih `learning_rate=0,03`, `max_depth=5`, `n_estimators=500` (belajar pelan dan dangkal).
- Aturan pemilihan model ditetapkan **sebelum** melihat test: model dengan CV PR-AUC tertinggi di data train → **Random Forest**. Karena selisihnya sangat kecil, XGBoost sama-sama layak; hal ini dijelaskan terbuka di bagian kesimpulan.

In [ ]:
# Model final dipilih dari hasil CV pada TRAIN (bukan dari test)
best_name = cv_table['CV PR-AUC'].idxmax()
print('Model terbaik menurut CV PR-AUC (train):', best_name)

## 2. Pemilihan threshold (out-of-fold pada data train)
Prediksi *out-of-fold* (5-fold) dari model hasil tuning dipakai untuk memilih threshold yang memaksimalkan **F1 kelas batal**. Threshold tidak ditentukan dari data test.

In [ ]:
skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof, thr = {}, {}
for name, s in searches.items():
    oof[name] = cross_val_predict(s.best_estimator_, X_train, y_train, cv=skf5, method='predict_proba')[:, 1]
    p, r, t = precision_recall_curve(y_train, oof[name])
    f1 = 2*p[:-1]*r[:-1] / (p[:-1] + r[:-1] + 1e-12)
    thr[name] = float(t[np.argmax(f1)])
    print(f'{name:13s} threshold terbaik (maks F1 OOF) = {thr[name]:.3f} | F1 OOF = {f1.max():.3f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
grid = np.linspace(0.05, 0.95, 91)
for ax, name in zip(axes, searches):
    f1s = [f1_score(y_train, oof[name] >= g) for g in grid]
    rec = [recall_score(y_train, oof[name] >= g) for g in grid]
    pre = [precision_score(y_train, oof[name] >= g, zero_division=0) for g in grid]
    ax.plot(grid, f1s, label='F1'); ax.plot(grid, rec, label='Recall'); ax.plot(grid, pre, label='Precision')
    ax.axvline(thr[name], color='k', ls='--', lw=1); ax.set_title(f'{name} (thr={thr[name]:.2f})'); ax.set_xlabel('threshold')
axes[0].legend(); plt.tight_layout(); plt.savefig('reports/figures/threshold_selection.png', dpi=120); plt.show()

**Interpretasi threshold:** threshold optimal (maksimum F1 pada prediksi out-of-fold di train) untuk ketiga model hampir sama, yaitu ≈0,30 (0,297 / 0,304 / 0,304), bukan 0,5. Wajar karena hanya ≈27% reservasi yang batal: dengan threshold 0,5 model terlalu "pelit" menandai pembatalan, sehingga Recall rendah. Pada kurva, menurunkan threshold menaikkan Recall tetapi menurunkan Precision; titik putus-putus menandai keseimbangan terbaik berdasarkan F1. Threshold ini dipilih **tanpa** menyentuh data test.

## 3. Melatih ulang pada seluruh data train
`best_estimator_` dari tuning sudah di-*refit* pada seluruh data train. Tahap ini hanya mengambil modelnya.

In [ ]:
final_models = {name: s.best_estimator_ for name, s in searches.items()}
print({k: type(v.named_steps['model']).__name__ for k, v in final_models.items()})

## 4. Evaluasi akhir pada data test (satu kali)

In [ ]:
def metrics(y, proba, threshold):
    pred = (proba >= threshold).astype(int)
    return {'Accuracy': accuracy_score(y, pred), 'Precision': precision_score(y, pred, zero_division=0),
            'Recall': recall_score(y, pred), 'F1': f1_score(y, pred),
            'ROC-AUC': roc_auc_score(y, proba), 'PR-AUC': average_precision_score(y, proba)}

proba_test  = {n: m.predict_proba(X_test)[:, 1]  for n, m in final_models.items()}
proba_train = {n: m.predict_proba(X_train)[:, 1] for n, m in final_models.items()}

rows = []
for n in final_models:
    for label, t in [('thr=0,5', 0.5), (f'thr={thr[n]:.2f} (OOF)', thr[n])]:
        rows.append({'model': n, 'threshold': label, **metrics(y_test, proba_test[n], t)})
test_table = pd.DataFrame(rows).set_index(['model', 'threshold']).round(3)
test_table.to_csv('reports/model_comparison.csv')
test_table

**Interpretasi hasil test:**
- **Threshold 0,5 vs ≈0,30:** pada Random Forest, Recall naik dari 0,526 menjadi 0,715 dan F1 dari 0,581 menjadi 0,622, dengan harga Precision turun (0,647 → 0,551) dan Accuracy turun (0,793 → 0,763). ROC-AUC dan PR-AUC tidak berubah karena tidak bergantung pada threshold.
- **Model terbaik:** Random Forest (F1 0,622; ROC-AUC 0,835; PR-AUC 0,645) dan XGBoost (0,621; 0,836; 0,647) **praktis setara**. **Terendah:** Logistic Regression (F1 0,599; ROC-AUC 0,804; PR-AUC 0,593). Alasan kemungkinan: model linear tidak menangkap interaksi antar fitur (misalnya `country` × `market_segment`), sedangkan model berbasis pohon bisa.
- Hasil test sangat dekat dengan hasil CV di train (PR-AUC RF 0,645 vs CV 0,647), jadi tidak ada tanda bahwa performa hanya kebetulan pada satu split.

In [ ]:
# Perbandingan train vs test pada threshold terpilih -> cek overfitting/underfitting
rows = []
for n in final_models:
    tr, te = metrics(y_train, proba_train[n], thr[n]), metrics(y_test, proba_test[n], thr[n])
    rows.append({'model': n, 'F1 train': tr['F1'], 'F1 test': te['F1'], 'selisih F1': tr['F1']-te['F1'],
                 'ROC-AUC train': tr['ROC-AUC'], 'ROC-AUC test': te['ROC-AUC'], 'selisih AUC': tr['ROC-AUC']-te['ROC-AUC'],
                 'PR-AUC train': tr['PR-AUC'], 'PR-AUC test': te['PR-AUC']})
gap = pd.DataFrame(rows).set_index('model').round(3); gap

**Interpretasi overfitting/underfitting (train vs test):**
- **Logistic Regression:** skor train dan test hampir sama (ROC-AUC 0,801 vs 0,804). Tidak overfitting, tetapi skornya paling rendah, menandakan kapasitas model terbatas (cenderung *underfitting* dibanding model pohon).
- **XGBoost:** selisih kecil (ROC-AUC 0,841 vs 0,836; PR-AUC 0,660 vs 0,647), generalisasi baik.
- **Random Forest:** selisih terbesar (ROC-AUC 0,853 vs 0,835; PR-AUC 0,686 vs 0,645), yaitu **overfitting ringan**. Masih dapat diterima karena performa test tetap setara model lain dan sama dengan hasil CV.

In [ ]:
# Confusion matrix (threshold terpilih) + kurva ROC dan PR
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, n in zip(axes, final_models):
    cm = confusion_matrix(y_test, proba_test[n] >= thr[n])
    ConfusionMatrixDisplay(cm, display_labels=['Tidak batal', 'Batal']).plot(ax=ax, colorbar=False, values_format='d')
    ax.set_title(f'{n} (thr={thr[n]:.2f})')
plt.tight_layout(); plt.savefig('reports/figures/confusion_matrices.png', dpi=120); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for n in final_models:
    fpr, tpr, _ = roc_curve(y_test, proba_test[n]); axes[0].plot(fpr, tpr, label=f"{n} (AUC={roc_auc_score(y_test, proba_test[n]):.3f})")
    p, r, _ = precision_recall_curve(y_test, proba_test[n]); axes[1].plot(r, p, label=f"{n} (AP={average_precision_score(y_test, proba_test[n]):.3f})")
axes[0].plot([0, 1], [0, 1], 'k--', lw=1); axes[0].set_title('ROC (test)'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[1].axhline(y_test.mean(), color='k', ls='--', lw=1); axes[1].set_title('Precision-Recall (test)'); axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[0].legend(); axes[1].legend(); plt.tight_layout(); plt.savefig('reports/figures/roc_pr_curves.png', dpi=120); plt.show()
for n in final_models:
    tn, fp, fn, tp = confusion_matrix(y_test, proba_test[n] >= thr[n]).ravel()
    print(f'{n:13s} TN={tn} FP={fp} FN={fn} TP={tp}')

**Interpretasi confusion matrix dan kurva (Random Forest, threshold ≈0,30, 17.431 data test):** TN 9.901, FP 2.774, FN 1.356, TP 3.400.
- Dari 4.756 pembatalan sebenarnya, model menemukan 3.400 (Recall 71,5%) dan **melewatkan 1.356 (False Negative)**.
- Dari 12.675 reservasi yang tidak batal, 2.774 (21,9%) salah ditandai berpotensi batal (**False Positive**).
- Sekitar 35% reservasi ditandai berpotensi batal, sedangkan base rate pembatalan 27%, harga dari mengejar Recall.
- Kurva ROC dan PR untuk Random Forest dan XGBoost hampir menempel, dan keduanya jelas di atas Logistic Regression. Kurva PR berada jauh di atas garis dasar (≈0,27), tanda model benar-benar belajar.
- Arti praktis: FN = peluang kehilangan peringatan dini (kamar kosong tak terduga); FP = peringatan palsu (tindak lanjut yang sebenarnya tidak perlu). Aplikasi harus menyatakan bahwa hasil hanya estimasi.

## 5. Analisis kesalahan dan pengaruh fitur (model terbaik)

In [ ]:
best = best_name
err = X_test.copy(); err['y'] = y_test.values; err['proba'] = proba_test[best]; err['pred'] = (err.proba >= thr[best]).astype(int)
err['FN'] = ((err.y == 1) & (err.pred == 0)).astype(int); err['FP'] = ((err.y == 0) & (err.pred == 1)).astype(int)
err['lead_bin'] = pd.cut(err.lead_time, [-1, 7, 30, 90, 180, 800], labels=['0-7', '8-30', '31-90', '91-180', '>180'])

def by(col):
    g = err.groupby(col, observed=True).agg(n=('y', 'size'), rasio_batal=('y', 'mean'), FN=('FN', 'sum'), FP=('FP', 'sum'))
    g['recall'] = err[err.y == 1].groupby(col, observed=True).pred.mean()
    return g.round(3)
print('Model analisis:', best)
for c in ['lead_bin', 'market_segment', 'customer_type']:
    print('\n==', c); display(by(c))

**Interpretasi analisis kesalahan (Random Forest):**
- **Berdasarkan lead time:** Recall hanya **10,3%** pada pemesanan 0-7 hari (270 dari 3.626 reservasi lolos), lalu naik menjadi 63% (8-30 hari), 74% (31-90), 76% (91-180), dan 89% (>180 hari). Model sulit mendeteksi pembatalan mendadak karena tidak ada fitur yang menandainya; pada pemesanan jauh hari justru banyak FP.
- **Berdasarkan market segment:** `Online TA` (10.369 baris) menyumbang FP terbanyak (2.104) dan FN terbanyak (959). Recall rendah pada `Complementary` (9,5%), `Corporate` (28,1%), dan `Direct` (32,7%); segmen ini memang jarang batal (≈13-15%) sehingga model jarang menandainya.
- **Berdasarkan customer type:** `Transient` mendominasi kesalahan (FN 1.228, FP 2.437) karena jumlahnya besar (14.400 baris).
- **Keterbatasan yang tampak:** model hanya melihat 5 input sehingga tidak tahu penyebab pembatalan sebenarnya; pembatalan mendadak dan segmen langka hampir tidak terdeteksi.

In [ ]:
# Permutation importance (pada sampel data test, metrik PR-AUC) -> pengaruh tiap INPUT
samp = X_test.sample(8000, random_state=RANDOM_STATE); ysamp = y_test.loc[samp.index]
pi = permutation_importance(final_models[best], samp, ysamp, scoring='average_precision', n_repeats=5, random_state=RANDOM_STATE, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=FEATS).sort_values(ascending=True)
ax = imp.plot(kind='barh', figsize=(7, 3.5)); ax.set_xlabel('Penurunan PR-AUC saat fitur diacak'); ax.set_title(f'Permutation importance ({best})')
plt.tight_layout(); plt.savefig('reports/figures/permutation_importance.png', dpi=120); plt.show()
imp.sort_values(ascending=False).round(4)

**Interpretasi permutation importance (penurunan PR-AUC saat fitur diacak):** `lead_time` (0,182) paling berpengaruh, disusul `country` (0,164), `market_segment` (0,161), `total_of_special_requests` (0,136), dan `customer_type` (0,063). Urutan ini konsisten dengan EDA di M3 (lead time dan segmen pemesanan membedakan rasio batal paling jelas). **Catatan:** ini menunjukkan fitur mana yang paling dipakai model, **bukan** bukti sebab-akibat; fitur bisa saling berkorelasi, sehingga pengaruh satu fitur dapat terbagi dengan fitur lain.

In [ ]:
# Kalibrasi probabilitas (model terbaik, data test)
pb = proba_test[best]
frac, mean_pred = calibration_curve(y_test, pb, n_bins=10, strategy='quantile')
plt.figure(figsize=(4.8, 4.5)); plt.plot(mean_pred, frac, 'o-', label=best); plt.plot([0, 1], [0, 1], 'k--', label='ideal')
plt.xlabel('probabilitas prediksi'); plt.ylabel('proporsi batal sebenarnya'); plt.title('Kurva kalibrasi (test)'); plt.legend()
plt.tight_layout(); plt.savefig('reports/figures/calibration.png', dpi=120); plt.show()
print('Brier score:', round(brier_score_loss(y_test, pb), 4), '| Brier baseline (selalu prediksi rata-rata):', round(brier_score_loss(y_test, np.full(len(y_test), y_train.mean())), 4))

**Interpretasi kalibrasi:** titik kurva sangat dekat dengan garis diagonal; misalnya prediksi rata-rata 0,46 bertepatan dengan proporsi batal sebenarnya 0,43, dan prediksi 0,73 dengan 0,74 (bin teratas). Brier score 0,1416 lebih baik dari baseline yang selalu menebak rata-rata (0,1984). Karena model dilatih **tanpa pembobotan kelas** dan trade-off diatur lewat threshold, probabilitas cukup terkalibrasi. Aplikasi boleh menampilkan probabilitas sebagai **estimasi** (tetap dengan disclaimer), dan kalibrasi tambahan (misalnya isotonic) tidak diperlukan.

## 6. Menyimpan model final

In [ ]:
final = {'pipeline': final_models[best], 'model_name': best, 'threshold': thr[best], 'features': FEATS,
         'metrics_test': {k: round(float(v), 4) for k, v in metrics(y_test, proba_test[best], thr[best]).items()},
         'random_state': RANDOM_STATE}
joblib.dump(final, 'models/final_model.joblib')
for n, m in final_models.items(): joblib.dump(m, f'models/tuned_{n.lower()}.joblib')

# Uji muat ulang + prediksi satu reservasi contoh (alur yang dipakai aplikasi)
loaded = joblib.load('models/final_model.joblib')
sample = pd.DataFrame([{'lead_time': 120, 'country': 'PRT', 'market_segment': 'Online TA',
                        'total_of_special_requests': 0, 'customer_type': 'Transient'}])
pr = loaded['pipeline'].predict_proba(sample[loaded['features']])[0, 1]
print(loaded['model_name'], '| threshold', round(loaded['threshold'], 3), '| P(batal) =', round(pr, 3),
      '->', 'Berpotensi Dibatalkan' if pr >= loaded['threshold'] else 'Tidak Berpotensi Dibatalkan')
print(json.dumps(loaded['metrics_test'], indent=2))

## Kesimpulan M6
1. **Model final:** Random Forest (hasil tuning), threshold ≈0,30. Dipilih dengan aturan yang ditetapkan sebelum evaluasi test: CV PR-AUC tertinggi pada data train.
2. **Hasil test (threshold ≈0,30):** Accuracy 0,763, Precision 0,551, Recall 0,715, F1 0,622, ROC-AUC 0,835, PR-AUC 0,645.
3. **Perbandingan jujur:** Random Forest dan XGBoost praktis setara (selisih PR-AUC < 0,003 pada CV dan test). XGBoost memiliki selisih train-test lebih kecil dan file model jauh lebih kecil (≈1 MB vs ≈37 MB); Random Forest dipilih karena aturan CV di atas. Keduanya dapat dipertanggungjawabkan; jika memilih XGBoost, sebutkan alasannya (generalisasi dan ukuran model), bukan hanya skor test.
4. **Keterbatasan:** pembatalan mendadak (lead time ≤7 hari) hampir tidak terdeteksi (Recall 10%); segmen langka lemah; hanya 5 input; `Non Refund` tidak masuk fitur; data 2015-2017 satu kawasan; ~28% FP pada reservasi tidak batal.
5. **Pengembangan lanjutan:** tambah fitur seperti `adr`/`hotel` (+≈0,02 ROC-AUC), uji kalibrasi isotonic, sesuaikan threshold dengan biaya bisnis nyata.
6. Model final tersimpan di `models/final_model.joblib` (berisi pipeline, threshold, daftar fitur, metrik test).

**Langkah berikutnya: M7 Aplikasi Streamlit** (`app.py` dan halaman evaluasi): form 5 input, validasi, prediksi dengan pipeline dan threshold tersimpan, tampilan hasil beserta probabilitas dan disclaimer, serta halaman evaluasi model. Pasang dulu: `pip install streamlit`.